# Windowed 16-QAM equalizer comparison

Пилотное сравнение MLP, 1D-CNN, двунаправленной LSTM и компактного Transformer на окнах из 7 I/Q-символов. Сеть регрессирует I/Q центрального переданного символа; BER считается после ближайшей точки созвездия.

Разбиение по целым PR-файлам: PR1–PR7 train, PR8 validation, PR9–PR10 test. Три инициализации на архитектуру. Все числа ниже относятся к одному каналу и являются пилотной оценкой, а не финальной статистической проверкой.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / 'reports' / 'results').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RESULTS = ROOT / 'reports' / 'results'
summary = pd.read_csv(RESULTS / 'model_comparison_summary.csv')
runs = pd.read_csv(RESULTS / 'model_comparison.csv')
inference = pd.read_csv(RESULTS / 'inference_benchmark.csv')
ffe = pd.read_csv(RESULTS / 'linear_ffe_baseline.csv')
display(summary)
display(ffe)


In [ ]:
raw = summary['test_raw_ber'].iloc[0]
relative_gain = 100 * (raw - summary['test_ber_mean']) / raw
relative_sd = 100 * summary['test_ber_std'] / raw
lat = inference.set_index('model')['single_window_latency_ms_median'].reindex(summary['model'])
plot_data = pd.concat([
    summary[['model', 'test_ber_mean', 'test_ber_std']],
    pd.DataFrame([{'model': 'Complex FFE', 'test_ber_mean': ffe['test_ber'].iloc[0], 'test_ber_std': 0.0}]),
], ignore_index=True)
plot_data['gain_pct'] = 100 * (raw - plot_data['test_ber_mean']) / raw
plot_data['gain_sd_pct'] = 100 * plot_data['test_ber_std'] / raw

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={'width_ratios': [1.15, 1]})
color_map = {'1D-CNN':'#60a5fa', 'LSTM':'#f59e0b', 'MLP':'#6b7280', 'Transformer':'#7c3aed', 'Complex FFE':'#b45309'}
colors = [color_map[name] for name in plot_data['model']]
ax1.bar(plot_data['model'], plot_data['gain_pct'], yerr=plot_data['gain_sd_pct'], capsize=4, color=colors)
ax1.axhline(0, color='#dc2626', linestyle='--', linewidth=1.4, label='raw BER baseline')
ax1.set_ylabel('BER reduction vs raw baseline (%)')
ax1.set_title('Test quality; neural bars: mean ± SD across 3 seeds')
ax1.legend(frameon=False)
ax1.grid(axis='y', alpha=.25)

neural_gain = summary.set_index('model').loc[inference['model'], 'test_ber_mean']
neural_gain = 100 * (raw - neural_gain.to_numpy()) / raw
ax2.scatter(lat, neural_gain, s=90, color=[color_map[name] for name in inference['model']])
for name, x, y in zip(inference['model'], lat, neural_gain):
    ax2.annotate(name, (x, y), xytext=(7, 5), textcoords='offset points', fontsize=9)
ax2.set_xscale('log')
ax2.set_xlim(lat.min() * 0.75, lat.max() * 2.0)
ax2.axhline(0, color='#dc2626', linestyle='--', linewidth=1.4)
ax2.set_xlabel('Median CPU latency for one window (ms, log scale)')
ax2.set_ylabel('BER reduction vs raw baseline (%)')
ax2.set_title('Neural model quality–latency trade-off')
ax2.grid(alpha=.25)
fig.suptitle('Pilot model comparison — held-out PR9–PR10', y=1.02, fontsize=13)
fig.tight_layout()
figure_dir = ROOT / 'reports' / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / 'model_tradeoff.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
inference[['model', 'parameters', 'single_window_latency_ms_median', 'batch_throughput_symbols_per_sec', 'checkpoint_seed', 'test_ber_checkpoint']].sort_values('test_ber_checkpoint')

In [ ]:
runs[['model', 'seed', 'val_ber', 'test_ber', 'test_ber_improvement_pct', 'train_seconds', 'best_epoch']].sort_values(['model', 'seed'])

## Интерпретация

В этом пилотном запуске трансформер показал минимальный BER. Он также имеет больше параметров, дольше обучается и медленнее выполняет вывод на CPU. Результат надо перепроверить на групповом кросс-валидировании по PR-файлам или новых реализациях. Не подбирать архитектуру по PR9–PR10 и одновременно называть ту же оценку независимой финальной проверкой.